# Retrain the EcBiz 0264 marker (Kaggle)

1. **Datasets -> New Dataset** -> upload `kaggle-training-pack.zip` (the ~11 MB training data + scripts).
2. **Import Notebook** -> choose `kaggle_marker_training.ipynb` (NOT the zip — Kaggle only accepts `.ipynb`/`.py` under 1 MB here).
3. In the notebook: **Add Data** -> select the dataset you created in step 1.
4. Optional: **Add-ons -> Secrets** -> add `HF_TOKEN` with your Hugging Face write token to auto-upload at the end.
5. Set **Accelerator: GPU T4 x2** (or T4 ×1 — both fit in 90 minutes), then run all cells. The setup cell always refreshes the scripts from your uploaded dataset.

In [ ]:
!pip -q uninstall -y diffusers gradio gradio-client optimum
!pip -q install -U "transformers>=4.44,<5" peft accelerate sentencepiece "optimum>=2.1,<3" "optimum-onnx>=0.1" onnx onnxruntime-gpu huggingface_hub

In [ ]:
import os, json, pathlib, shutil
root = pathlib.Path('/kaggle/working')
inp = pathlib.Path('/kaggle/input')
sources = [p.parent.parent for p in inp.rglob('train.jsonl') if p.parent.name == 'dataset'] if inp.exists() else []
if not sources:
    raise RuntimeError('Add the kaggle-training-pack dataset using the Add Data button.')
src = sources[0]
for name in ('kaggle_train.py', 'export_onnx.py', 'prepare_dataset.py', 'upload_to_hf.py', 'chat_template.jinja', 'README.md'):
    item = src / name
    if item.exists():
        shutil.copy2(item, root / name)
if not (root / 'dataset' / 'train.jsonl').exists():
    shutil.copytree(src / 'dataset', root / 'dataset', dirs_exist_ok=True)
print('train examples:', len((root / 'dataset' / 'train.jsonl').read_text(encoding='utf-8').splitlines()))
print(json.loads((root / 'dataset' / 'coverage.json').read_text(encoding='utf-8')))

Train on **both T4s**. Make sure Kaggle accelerator is set to **GPU T4 x2**. If launching fails for any reason, replace this cell with `!python kaggle_train.py` to use a single T4 (still fits in 90 minutes).

In [ ]:
!torchrun --nproc_per_node=2 kaggle_train.py

In [ ]:
!python export_onnx.py

## Optional: upload to Hugging Face

Add a Kaggle secret named `HF_TOKEN` (Add-ons -> Secrets) with a **write** token. This cell uploads to `SemiLunarz/qwen3-0.6b-igcse-0264-marker-v2` by default; set `HF_REPO` to change it.

In [ ]:
import os, subprocess, sys
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token loaded from Kaggle secret')
except Exception as exc:
    print('No Kaggle secret HF_TOKEN:', exc)
if os.environ.get('HF_TOKEN'):
    subprocess.run([sys.executable, 'upload_to_hf.py'], check=True)
else:
    print('Skipping upload. Download output/onnx and upload manually.')

## Next

Update `MODEL_ID` (and `dtype` if the export is unquantised) in `subjects/business/js/views/ai-marker.js`, clear the cached model, then smoke-test the AI Marker page.